In [72]:
import sys, os
sys.path.append(os.path.dirname('__file__'))
sys.path.append(os.getcwd())
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
import pickle 
import numpy as np
sys.path.append(os.path.dirname(os.getcwd()))
from pyfrechet.metrics import mse
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from joblib import Parallel, delayed

from pyfrechet.metric_spaces import MetricData, LogEuclidean
from pyfrechet.metric_spaces import MetricData, LogEuclidean
from pyfrechet.regression.bagged_regressor import BaggedRegressor
from pyfrechet.regression.trees import Tree
from pyfrechet.metric_spaces.utils import vectorize, devectorize


# By-blocks execution
n_samples = len(os.listdir(os.path.join(os.getcwd(), 'simulations_SPD/data')))
n_cores = 4  # Adjusted for your MacBook Air (4 physical cores)
n_blocks = int(np.ceil(n_samples / n_cores))
current_block = 1

# Define parameter grid for tuning
param_grid = {
    'estimator__min_split_size': [1]
}

# Custom scorer (negative mean squared error)
neg_mse = make_scorer(mse, greater_is_better=False)

def tune_forest(X, y):
    """ Perform hyperparameter tuning using GridSearchCV. """
    base = Tree(split_type='2means', mtry=None, impurity_method='cart')
    forest = BaggedRegressor(estimator=base, n_estimators=100, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)
    
    tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=5, n_jobs=1, verbose=4)
    tuned_forest.fit(X, y)
    return tuned_forest.best_estimator_

with open(os.path.join(os.getcwd(), 'simulations_SPD/data/' + 'SPD_samp2_N100_df15_block_1.pkl'), 'rb') as f:
    sample = pickle.load(f)

X = np.c_[sample['t']]
sample_Y = np.array(sample['y'])

for dist in ['AI']:
    if dist == 'AI':
        M = LogEuclidean(dim=2)
        # To do: turn SPDVectorizer into a function, it is not necessary to define a class
        y_vectors = vectorize(sample_Y)
        y = MetricData(M, y_vectors)

        # Define the base tree and forest
        base = Tree(split_type='2means', impurity_method='cart', mtry=None, min_split_size=1)
        forest = BaggedRegressor(estimator=base, n_estimators=100, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)

        # Perform hyperparameter tuning
        param_grid = {'estimator__min_split_size': [1, 5]}
        neg_mse = make_scorer(mse, greater_is_better=False)

        tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=5, n_jobs=1, verbose=4)
        tuned_forest.fit(X, y)  # Use vectorized data

        best_forest = tuned_forest.best_estimator_
        best_forest.fit(X, y)  # Refit with best params

        results = {
            'x_train_data': X,
            'y_train_data': y_vectors,
            'train_predictions': best_forest.predict(X),
            'forest': best_forest,
        }
results
print(f'Block number: {current_block}')

Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV 1/5] END ......estimator__min_split_size=1;, score=-0.540 total time=   8.7s


/Users/Diego/miniconda3/envs/pballs/lib/python3.9/site-packages/geomstats/geometry/riemannian_metric.py:298: RuntimeWarning: invalid value encountered in sqrt
  dist = gs.sqrt(sq_dist)
/Users/Diego/miniconda3/envs/pballs/lib/python3.9/site-packages/geomstats/geometry/riemannian_metric.py:298: RuntimeWarning: invalid value encountered in sqrt
  dist = gs.sqrt(sq_dist)


[CV 2/5] END ......estimator__min_split_size=1;, score=-0.484 total time=   8.5s
[CV 3/5] END ......estimator__min_split_size=1;, score=-0.669 total time=   8.4s


/Users/Diego/miniconda3/envs/pballs/lib/python3.9/site-packages/geomstats/geometry/riemannian_metric.py:298: RuntimeWarning: invalid value encountered in sqrt
  dist = gs.sqrt(sq_dist)


[CV 4/5] END ......estimator__min_split_size=1;, score=-0.381 total time=   8.5s
[CV 5/5] END ......estimator__min_split_size=1;, score=-0.623 total time=   8.3s
[CV 1/5] END ......estimator__min_split_size=5;, score=-0.551 total time=   4.2s
[CV 2/5] END ......estimator__min_split_size=5;, score=-0.454 total time=   4.2s
[CV 3/5] END ......estimator__min_split_size=5;, score=-0.622 total time=   4.3s
[CV 4/5] END ......estimator__min_split_size=5;, score=-0.419 total time=   4.3s
[CV 5/5] END ......estimator__min_split_size=5;, score=-0.545 total time=   4.2s
Block number: 1


In [74]:
best_forest.oob_predict(X).data

array([[ 1.06332142,  0.50909517, -0.62971908],
       [ 0.80714061,  0.964301  ,  0.21568912],
       [ 0.71845214,  1.00687173,  0.32567576],
       [ 1.04103798,  0.96870143, -0.09752   ],
       [ 1.23205499,  0.55493713, -0.51533497],
       [ 0.95206161,  0.9800404 , -0.04821334],
       [ 0.94390857,  1.01474493, -0.02808335],
       [ 1.15147024,  0.53540773, -0.67205416],
       [ 1.15389657,  0.82829781, -0.15973527],
       [ 0.90740058,  0.9994558 ,  0.00642669],
       [ 0.78160707,  0.83541533,  0.13227833],
       [ 0.5011695 ,  0.83232661,  0.35056189],
       [ 0.98258553,  0.99752305, -0.09091418],
       [ 1.10278   ,  0.82424024, -0.18935898],
       [ 0.70295973,  0.98327081,  0.28670052],
       [ 0.9557478 ,  0.96307615, -0.13068657],
       [ 0.97198498,  0.94733989, -0.09365347],
       [ 0.75343263,  1.02531705,  0.37803259],
       [ 1.05940326,  1.00196063, -0.10069199],
       [ 1.25261226,  0.6252469 , -0.7847628 ],
       [ 1.16632103,  0.55740579, -0.697

In [75]:
count = 0
assert len(best_forest.estimators) > 0, "At least one estimator is needed to compute weights"
# Index of the observation x in the training data
if best_forest.X_train_.shape[1] == 1:
    x_idx = np.argwhere(np.isclose(best_forest.X_train_, X[1,:], rtol=1e-10).flatten()).flatten()
else:
    x_idx = np.argwhere(np.all(np.isclose(best_forest.X_train_, X[1,:], rtol=1e-10), axis=1)).flatten()
if len(x_idx) == 0:
    x_idx = None
    warnings.warn('Observation not found in training data, working with the full training set')
else:
    x_idx = x_idx[0]

print(f'x_idx: {x_idx}')
weights = np.zeros(best_forest.y_train_.shape[0])
for (mask, estimator) in best_forest.estimators:
    # Weights for non-OOB observations are zero, so that they do not have an effect over the prediction
    print(mask)
    if x_idx in mask:
        est_weights = np.repeat(0, best_forest.y_train_.shape[0])
    else:
        print('si')
        # Note that each base estimator has its own .weights_for() method
        est_weights = estimator.weights_for(X[1,:])
        #Count how many trees there are for which x is OOB. Notice that now, we aggregate only the weights of these trees, not over all the trees. 
        count += 1
    weights[mask] += est_weights
best_forest._normalize_weights(weights / count, clip=True)

x_idx: 1
[55 44 32 39 10 32 48 78 17 96  0  1 48 98 44 29 51 62 60 11 74 23 74 77
 10 12 46 11 95 90 44 33  0 93 62 39 69  1 62  4 76 14 63 21 72 96 25 95
 83 15 12 13 58  8 50 91 26 88 19 40 18 59 86 25 88 40 18 76  8 13 89 16
 57 58 64 39 24 83 69 92 60 14 96 93 51 61 57 55 66 34 81 75 22 59 16 10
 62 80 83 40]
[89 82 59 60 15 92  8 88  3 61 75 36 14 77 21 74  3 36 33 25 14 18 60 92
 22 72 83 23 23 38 69 55 67 46  5 49 41 17 51 28 67 26 70  9 47  6 17 24
 35 57  1 37 72 44 21 85 28 19  2 77 86 73 33 88 55 37 47 50 38 66 81 27
  0 45 63  7 86 55 19 27 51 68 50  2 75 46 61  3 64 41 19 64 93 20 47 13
 56 64 19 66]
[72 77 79  8 72 67 98 14  6 36  6 30 24 18 16 65 87 20 82 99  5 67 40 88
 22 50 47 32 15 35 14 68 17 16 56 50 13  2  4 85 11 52 28 54 15 18 55 46
 19  8 16 53 70  8 59 37 63 88 25 13 45 39 40 63 10 22 37 96 53 86 69 14
 36 92  6  1 31 46 12 24 86 69 83 53 84 41 64  0  4 44 99 19  5 10 51 66
 56 27  1 27]
[43 68 21 42 90 62 29  9 48 33 66 41  0 89 44 41 34 89  3 30  7 45 64 89


array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.05242266, 0.        , 0.        , 0.        , 0.02442562,
       0.        , 0.        , 0.        , 0.00195312, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.02829467, 0.        , 0.09209556, 0.        , 0.        ,
       0.07078937, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00479403, 0.        , 0.        , 0.        , 0.00359786,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.00359786, 0.        , 0.02147981,
       0.        , 0.03761775, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.00359786,
       0.        , 0.        , 0.        , 0.07863907, 0.        ,
       0.00195312, 0.0681671 , 0.00195312, 0.        , 0.     

In [76]:
best_forest.oob_weights_for(X[1,:])

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.05242266, 0.        , 0.        , 0.        , 0.02442562,
       0.        , 0.        , 0.        , 0.00195312, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.02829467, 0.        , 0.09209556, 0.        , 0.        ,
       0.07078937, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00479403, 0.        , 0.        , 0.        , 0.00359786,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.00359786, 0.        , 0.02147981,
       0.        , 0.03761775, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.00359786,
       0.        , 0.        , 0.        , 0.07863907, 0.        ,
       0.00195312, 0.0681671 , 0.00195312, 0.        , 0.     

In [77]:
best_forest.y_train_.frechet_mean(best_forest.oob_weights_for(X[1,:]))

array([0.80714061, 0.964301  , 0.21568912])

In [82]:
best_forest.oob_errors()

array([0.71498237, 1.06001623, 0.42334477, 0.64378005, 1.04319998,
       0.69906793, 0.52087969, 0.49353116, 0.34896867, 0.84472766,
       1.02605459, 0.37982053, 0.55222405, 0.5990804 , 0.60288662,
       1.04993602, 0.61440785, 0.55531326, 1.37295527, 0.43772485,
       0.76778582, 0.44626005, 0.43173892, 0.37286437, 0.86918756,
       0.53812506, 0.50667539, 0.8057555 , 0.98138209, 0.61310231,
       0.40526791, 1.43941474, 0.70132118, 0.64946328, 0.58708378,
       0.49670746, 0.62740453, 0.41436176, 0.49585889, 0.30152071,
       1.12953996, 0.99932073, 0.56430434, 0.44697788, 0.57813751,
       0.72898786, 0.76425222, 0.82486953, 0.61569454, 0.48074891,
       1.34299434, 0.58716974, 0.31235148, 0.50594692, 1.22219188,
       1.0853537 , 1.01263006, 0.73399308, 0.83074544, 0.42133295,
       0.46168323, 0.97670917, 0.48797355, 0.42499111, 0.73357038,
       0.38462056, 0.23304158, 0.84408854, 0.26283212, 0.9522038 ,
       0.2943385 , 0.74811505, 0.36272464, 0.57394921, 0.61669

In [49]:
best_forest._oob_predict_one(X[1,:])

array([nan, nan, nan])

In [79]:
# Check if estimator has been fitted (.fit() has been applied), otherwise raise an error
from sklearn.utils.validation import check_array, check_is_fitted

check_is_fitted(best_forest)
X = check_array(X) 

# To allow (1,p) and (,p) shapes ('matrices' and 'vectors')
y0 = best_forest._oob_predict_one(X[0,:])
# The shape of predictions will have x.shape[0] (number of observations for prediction) rows
# and y0.shape[0] (length (dimension) of the MetricData class we are handling) columns
y_pred = np.zeros((X.shape[0], y0.shape[0]))
y_pred[0,:] = y0
for i in range(1, X.shape[0]):
    y_pred[i,:] = best_forest._oob_predict_one(X[i,:])
print(MetricData(best_forest.y_train_.M, y_pred).data)

[[ 1.06332142  0.50909517 -0.62971908]
 [ 0.80714061  0.964301    0.21568912]
 [ 0.71845214  1.00687173  0.32567576]
 [ 1.04103798  0.96870143 -0.09752   ]
 [ 1.23205499  0.55493713 -0.51533497]
 [ 0.95206161  0.9800404  -0.04821334]
 [ 0.94390857  1.01474493 -0.02808335]
 [ 1.15147024  0.53540773 -0.67205416]
 [ 1.15389657  0.82829781 -0.15973527]
 [ 0.90740058  0.9994558   0.00642669]
 [ 0.78160707  0.83541533  0.13227833]
 [ 0.5011695   0.83232661  0.35056189]
 [ 0.98258553  0.99752305 -0.09091418]
 [ 1.10278     0.82424024 -0.18935898]
 [ 0.70295973  0.98327081  0.28670052]
 [ 0.9557478   0.96307615 -0.13068657]
 [ 0.97198498  0.94733989 -0.09365347]
 [ 0.75343263  1.02531705  0.37803259]
 [ 1.05940326  1.00196063 -0.10069199]
 [ 1.25261226  0.6252469  -0.7847628 ]
 [ 1.16632103  0.55740579 -0.6979324 ]
 [ 1.1743313   0.71346626 -0.18899766]
 [ 1.04922604  0.51908046 -0.63296   ]
 [ 0.95651005  0.99485231 -0.01049006]
 [ 1.08572668  0.53163832 -0.65703035]
 [ 0.76943825  0.94463578